In [1]:
import pandas as pd

ops = pd.read_csv('operation_history.csv')
ops['date_created'] = pd.to_datetime(ops['date_created'], errors='coerce')

In [2]:
# Общее число операций в истории.
# Показывает общий объем операционной активности и нагрузку на процесс.

total_operations = len(ops)
print(f"total_operations: {float(total_operations)}")

total_operations: 115.0


In [3]:
# Число уникальных грузомест.
# Базовый показатель фактического объема обработки.

unique_cargo_places = ops['cargo_place_uuid'].nunique()
print(f"unique_cargo_places: {float(unique_cargo_places)}")

unique_cargo_places: 5.0


In [4]:
# Число уникальных документов, участвующих в операциях.
# Характеризует интенсивность документооборота и сложность процесса.

unique_documents = ops['document_uuid'].nunique()
print(f"unique_documents: {float(unique_documents)}")

unique_documents: 80.0


In [5]:
# Число уникальных офисов, где были операции.
# Отражает ширину логистической сети и распределенность процесса.

unique_offices_involved = ops['current_office_uuid'].dropna().nunique()
print(f"unique_offices_involved: {float(unique_offices_involved)}")

unique_offices_involved: 15.0


In [ ]:
# Среднее число операций на одно грузоместо.
# Индикатор "трения" процесса, указывает на эффективность.

avg_operations_per_cargo_place = float(total_operations / unique_cargo_places)
print(f"avg_operations_per_cargo_place: {avg_operations_per_cargo_place}")

avg_operations_per_cargo_place: 23.0


In [8]:
# Средняя длительность жизненного цикла грузоместа (в часах).
# Контроль операционной скорости

timeline_by_place = (
    ops.groupby('cargo_place_uuid', as_index=False)
    .agg(first_event=('date_created', 'min'), last_event=('date_created', 'max'))
)

timeline_by_place['cycle_hours'] = (
    (timeline_by_place['last_event'] - timeline_by_place['first_event']).dt.total_seconds() / 3600
)

q1 = timeline_by_place['cycle_hours'].quantile(0.25)
q3 = timeline_by_place['cycle_hours'].quantile(0.75)
iqr = q3 - q1
high = q3 + 1.5 * iqr

timeline_no_outliers = timeline_by_place[timeline_by_place['cycle_hours'] <= high]
avg_cargo_lifecycle_hours = float(timeline_no_outliers['cycle_hours'].mean())


print(f"avg_cargo_lifecycle_hours: {avg_cargo_lifecycle_hours:.2f}")

avg_cargo_lifecycle_hours: 79.54


In [9]:
# Офис с максимальным количеством операций, количество операций в самом нагруженном офисе.
# Позволяют найти узкие места и точки перегрузки для балансировки ресурсов.

office_load = (
    ops[ops['current_office_uuid'].notna()]
    .groupby('current_office_uuid', as_index=False)
    .size()
    .rename(columns={'size': 'ops_cnt'})
    .sort_values('ops_cnt', ascending=False)
)

top_office = office_load.iloc[0]
print(f"top_office_by_operations: {top_office['current_office_uuid']}")
print(f"top_office_operations_count: {float(top_office['ops_cnt'])}")

top_office_by_operations: 38090d9d-dbc3-4a39-be44-b76cc7a4c168
top_office_operations_count: 26.0


In [10]:
# Самый частый тип операции.
# Помогает определить приоритет автоматизации и контрольных проверок.

operation_mix = ops.groupby('operation_type', as_index=False).size().rename(columns={'size': 'cnt'})
top_operation = operation_mix.sort_values('cnt', ascending=False).head(1)
print(f"top_operation_type: {top_operation.iloc[0]['operation_type']}")

top_operation_type: DOCUMENT_ADD_CARGO_PLACE
